# 🧠 MRI Brain Tumor Detection & Segmentation
### YOLOv8n-seg · Roboflow Dataset · Google Colab

| Detail | Value |
|---|---|
| **Model** | `yolov8n-seg.pt` (nano segmentation) |
| **Task** | Instance Segmentation (polygon masks) |
| **Dataset** | Roboflow — `my-dataset-ffjxl / convertion-mt8pg` |
| **Images** | 500 annotated MRI scans |
| **Classes** | Brain Tumor regions |

---
> **Before running:** Add your `ROBOFLOW_API_KEY` in **Colab → 🔑 Secrets** (left sidebar)

## 📦 Cell 1 — Install Dependencies

In [ ]:
!pip install roboflow ultralytics opencv-python-headless matplotlib pyyaml -q
print('✅ All dependencies installed')

## 🔑 Cell 2 — Load API Key from Colab Secrets

In [ ]:
from google.colab import userdata

ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')

if not ROBOFLOW_API_KEY:
    raise ValueError('❌ ROBOFLOW_API_KEY not found. Add it in Colab → 🔑 Secrets')

print('✅ Roboflow API key loaded successfully')

## 📥 Cell 3 — Download Dataset from Roboflow

In [ ]:
from roboflow import Roboflow

rf      = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace('my-dataset-ffjxl').project('convertion-mt8pg')
version = project.version(1)   # ← bump if you have v2, v3 etc.

# Export as yolov8 segmentation format
dataset = version.download('yolov8-seg')

DATA_YAML = f'{dataset.location}/data.yaml'
print(f'✅ Dataset saved to : {dataset.location}')
print(f'📄 data.yaml path   : {DATA_YAML}')

## 🔍 Cell 4 — Inspect data.yaml & Dataset Structure

In [ ]:
import yaml, os

with open(DATA_YAML, 'r') as f:
    cfg = yaml.safe_load(f)

print('='*45)
print('📦  DATASET CONFIGURATION')
print('='*45)
print(f'  Classes ({cfg["nc"]})  : {cfg["names"]}')
print(f'  Train path     : {cfg.get("train", "N/A")}')
print(f'  Val path       : {cfg.get("val",   "N/A")}')
print(f'  Test path      : {cfg.get("test",  "N/A")}')
print('='*45)

# Count images per split
for split in ['train', 'valid', 'test']:
    img_dir = os.path.join(dataset.location, split, 'images')
    if os.path.exists(img_dir):
        n = len(os.listdir(img_dir))
        print(f'  {split:6s} images : {n}')

## 🖼️ Cell 5 — Visualise Sample Annotated MRI Scans

In [ ]:
import glob, random
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import matplotlib.patches as patches
import numpy as np
import cv2

train_imgs = glob.glob(f'{dataset.location}/train/images/*.jpg') + \
             glob.glob(f'{dataset.location}/train/images/*.png')

random.seed(42)
samples = random.sample(train_imgs, min(8, len(train_imgs)))

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
fig.patch.set_facecolor('#0d0d0d')
fig.suptitle('🧠  MRI Brain Tumor — Annotated Training Samples',
             color='white', fontsize=16, fontweight='bold', y=1.01)

for ax, img_path in zip(axes.flatten(), samples):
    # Load image
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Load corresponding label (polygon mask)
    lbl_path = img_path.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
    h, w = img.shape[:2]

    overlay = img.copy()
    if os.path.exists(lbl_path):
        with open(lbl_path) as lf:
            for line in lf:
                parts = list(map(float, line.strip().split()))
                cls   = int(parts[0])
                coords = np.array(parts[1:]).reshape(-1, 2)
                coords[:, 0] *= w
                coords[:, 1] *= h
                pts = coords.astype(np.int32)
                cv2.polylines(overlay, [pts], True, (57, 255, 20), 2)
                cv2.fillPoly(overlay, [pts], (57, 255, 20))
        img = cv2.addWeighted(img, 0.65, overlay, 0.35, 0)

    ax.imshow(img, cmap='gray')
    ax.set_title(os.path.basename(img_path), color='#aaaaaa', fontsize=8)
    ax.axis('off')
    ax.set_facecolor('#0d0d0d')

plt.tight_layout()
plt.savefig('/content/sample_annotations.png', dpi=120,
            bbox_inches='tight', facecolor='#0d0d0d')
plt.show()
print('✅ Sample grid saved → /content/sample_annotations.png')

## ⚙️ Cell 6 — GPU Check

In [ ]:
import torch

cuda_ok = torch.cuda.is_available()
device  = 0 if cuda_ok else 'cpu'

print('='*40)
if cuda_ok:
    print(f'  ✅ GPU  : {torch.cuda.get_device_name(0)}')
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  💾 VRAM : {mem:.1f} GB')
else:
    print('  ⚠️  No GPU found — using CPU (slow)')
    print('  → Runtime → Change runtime type → T4 GPU')
print('='*40)

## 🚀 Cell 7 — Train YOLOv8n-seg (Instance Segmentation)

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv8 nano SEGMENTATION model
model = YOLO('yolov8n-seg.pt')

print('🧠 Starting MRI Brain Tumor Segmentation Training...')
print('='*55)

results = model.train(
    data      = DATA_YAML,           # your Roboflow data.yaml
    epochs    = 100,                 # increase if val loss still dropping
    imgsz     = 640,                 # standard YOLO input size
    batch     = 16,                  # reduce to 8 if OOM error
    name      = 'mri_tumor_seg',
    project   = '/content/runs/segment',
    device    = device,
    workers   = 2,                   # keep low in Colab
    patience  = 20,                  # early stopping patience
    lr0       = 0.01,                # initial learning rate
    lrf       = 0.001,               # final lr (cosine decay)
    momentum  = 0.937,
    weight_decay = 0.0005,
    warmup_epochs = 3,
    augment   = True,
    mosaic    = 1.0,                 # mosaic augmentation
    degrees   = 10.0,                # rotation (MRI safe range)
    flipud    = 0.5,                 # vertical flip
    fliplr    = 0.5,                 # horizontal flip
    scale     = 0.5,                 # scale jitter
    hsv_h     = 0.0,                 # no hue shift (grayscale MRI)
    hsv_s     = 0.0,                 # no saturation shift
    hsv_v     = 0.4,                 # brightness variation only
    save      = True,
    plots     = True,
    exist_ok  = True,
    verbose   = True,
)

BEST_WEIGHTS = '/content/runs/segment/mri_tumor_seg/weights/best.pt'
print(f'\n✅ Training complete!')
print(f'📂 Best weights → {BEST_WEIGHTS}')

## 📊 Cell 8 — Training Curve Plots

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

RUN_DIR = '/content/runs/segment/mri_tumor_seg'

plot_files = {
    'results.png'       : 'Training & Validation Metrics',
    'confusion_matrix.png' : 'Confusion Matrix',
    'PR_curve.png'      : 'Precision-Recall Curve',
    'F1_curve.png'      : 'F1 Score Curve',
}

for fname, title in plot_files.items():
    fpath = f'{RUN_DIR}/{fname}'
    if os.path.exists(fpath):
        fig, ax = plt.subplots(figsize=(12, 6))
        ax.imshow(mpimg.imread(fpath))
        ax.set_title(f'📈  {title}', fontsize=14, fontweight='bold')
        ax.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print(f'⚠️  {fname} not found yet')

## ✅ Cell 9 — Evaluate on Validation Set

In [ ]:
from ultralytics import YOLO

best_model = YOLO(BEST_WEIGHTS)
metrics    = best_model.val(data=DATA_YAML, device=device)

print('\n' + '='*50)
print('  🧠  MRI TUMOR SEGMENTATION — VAL METRICS')
print('='*50)
print(f'  Box  mAP@50        : {metrics.box.map50:.4f}')
print(f'  Box  mAP@50-95     : {metrics.box.map:.4f}')
print(f'  Box  Precision     : {metrics.box.mp:.4f}')
print(f'  Box  Recall        : {metrics.box.mr:.4f}')
print('  ─────────────────────────────────────')
print(f'  Mask mAP@50        : {metrics.seg.map50:.4f}')
print(f'  Mask mAP@50-95     : {metrics.seg.map:.4f}')
print(f'  Mask Precision     : {metrics.seg.mp:.4f}')
print(f'  Mask Recall        : {metrics.seg.mr:.4f}')
print('='*50)

## 🔬 Cell 10 — Predict on Test MRI Scans

In [ ]:
test_img_dir = f'{dataset.location}/test/images'

# Fallback to valid if no test split
if not os.path.exists(test_img_dir) or len(os.listdir(test_img_dir)) == 0:
    test_img_dir = f'{dataset.location}/valid/images'
    print('ℹ️  No test split found — using validation images')

test_results = best_model.predict(
    source      = test_img_dir,
    conf        = 0.25,          # confidence threshold
    iou         = 0.45,          # NMS IoU threshold
    save        = True,
    save_txt    = True,          # save masks as .txt
    show_labels = True,
    show_conf   = True,
    retina_masks= True,          # high-quality masks
    device      = device,
)

print(f'✅ Predictions saved — total images: {len(test_results)}')

## 🎨 Cell 11 — Visualise Segmentation Predictions

In [ ]:
import glob
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np

pred_dir   = '/content/runs/segment/predict'
pred_images = sorted(glob.glob(f'{pred_dir}/*.jpg') +
                     glob.glob(f'{pred_dir}/*.png'))[:8]

if not pred_images:
    # Ultralytics sometimes names the folder predict2, predict3 etc.
    pred_dir = sorted(glob.glob('/content/runs/segment/predict*'))[-1]
    pred_images = sorted(glob.glob(f'{pred_dir}/*.jpg') +
                         glob.glob(f'{pred_dir}/*.png'))[:8]

rows = (len(pred_images) + 3) // 4
fig, axes = plt.subplots(rows, 4, figsize=(20, rows * 5))
fig.patch.set_facecolor('#0d0d0d')
fig.suptitle('🧠  YOLOv8n-seg  ·  MRI Brain Tumor Segmentation Predictions',
             color='white', fontsize=16, fontweight='bold', y=1.01)

for i, ax in enumerate(axes.flatten()):
    if i < len(pred_images):
        img = mpimg.imread(pred_images[i])
        ax.imshow(img)
        ax.set_title(os.path.basename(pred_images[i]),
                     color='#39ff14', fontsize=9, fontweight='bold')
    ax.axis('off')
    ax.set_facecolor('#0d0d0d')

plt.tight_layout()
plt.savefig('/content/predictions_grid.png', dpi=150,
            bbox_inches='tight', facecolor='#0d0d0d')
plt.show()
print('✅ Prediction grid saved → /content/predictions_grid.png')

## 🔎 Cell 12 — Per-Image Detection Report

In [ ]:
print('\n🧠  MRI TUMOR DETECTION REPORT')
print('='*65)

total_tumors = 0
for r in test_results:
    fname   = os.path.basename(r.path)
    n_dets  = len(r.boxes)
    total_tumors += n_dets

    if n_dets == 0:
        print(f'  📷 {fname:30s}  → No tumor detected')
    else:
        for box in r.boxes:
            cls_id = int(box.cls[0])
            cls_nm = best_model.names[cls_id]
            conf   = float(box.conf[0])
            x1,y1,x2,y2 = box.xyxy[0].tolist()
            area = (x2 - x1) * (y2 - y1)
            print(f'  📷 {fname:30s}  '
                  f'class={cls_nm:12s}  '
                  f'conf={conf:.2f}  '
                  f'area={area:.0f}px²')

print('='*65)
print(f'  Total detections : {total_tumors}')
print(f'  Total images     : {len(test_results)}')
positive = sum(1 for r in test_results if len(r.boxes) > 0)
print(f'  Images w/ tumor  : {positive} / {len(test_results)}')
print('='*65)

## 💾 Cell 13 — Export Model (ONNX + TorchScript)

In [ ]:
# ONNX — for deployment (FastAPI, ONNX Runtime, TensorRT)
best_model.export(format='onnx', dynamic=True, simplify=True)
print('✅ ONNX model exported')

# TorchScript — for PyTorch Mobile / C++ inference
best_model.export(format='torchscript')
print('✅ TorchScript model exported')

## 📥 Cell 14 — Download Weights to Local PC

In [ ]:
from google.colab import files
import shutil

# Zip run folder (weights + plots + metrics)
shutil.make_archive('/content/mri_tumor_run',
                    'zip',
                    '/content/runs/segment/mri_tumor_seg')

# Download best.pt directly
files.download(BEST_WEIGHTS)

# Download full run zip
files.download('/content/mri_tumor_run.zip')

print('✅ Files sent for download!')

## 🔁 Cell 15 — Inference on Any New MRI Image
> Upload any MRI scan and run this cell to get the tumor mask

In [ ]:
from google.colab import files as colab_files
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import cv2, numpy as np

# Upload your MRI image
uploaded = colab_files.upload()
img_path = list(uploaded.keys())[0]

# Run prediction
r = best_model.predict(
    source       = img_path,
    conf         = 0.25,
    iou          = 0.45,
    retina_masks = True,
    save         = True,
    device       = device,
)[0]

# Show original vs prediction side by side
orig = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)

# Get rendered prediction image from result
pred_img = r.plot(labels=True, conf=True, masks=True)
pred_img = cv2.cvtColor(pred_img, cv2.COLOR_BGR2RGB)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('#0d0d0d')

ax1.imshow(orig, cmap='gray')
ax1.set_title('Original MRI', color='white', fontsize=13, fontweight='bold')
ax1.axis('off')

ax2.imshow(pred_img)
ax2.set_title(f'YOLOv8n-seg Prediction  [{len(r.boxes)} detection(s)]',
              color='#39ff14', fontsize=13, fontweight='bold')
ax2.axis('off')

plt.tight_layout()
plt.savefig('/content/single_prediction.png', dpi=150,
            bbox_inches='tight', facecolor='#0d0d0d')
plt.show()

# Print detections
print(f'\n🧠  Detections in {img_path}:')
for box in r.boxes:
    cls  = best_model.names[int(box.cls[0])]
    conf = float(box.conf[0])
    print(f'   → {cls}  confidence: {conf:.2%}')

---
## 📋 Summary

| Step | Done |
|---|---|
| Install dependencies | ✅ |
| Load Roboflow API key from Secrets | ✅ |
| Download annotated MRI dataset | ✅ |
| Inspect data.yaml + class info | ✅ |
| Visualise annotated samples | ✅ |
| Train `yolov8n-seg.pt` | ✅ |
| Plot training curves | ✅ |
| Evaluate — box + mask mAP | ✅ |
| Predict on test MRI scans | ✅ |
| Visualise segmentation masks | ✅ |
| Per-image detection report | ✅ |
| Export ONNX + TorchScript | ✅ |
| Download weights | ✅ |
| Single-image inference tool | ✅ |

### 🔧 Quick tuning tips
| Issue | Fix |
|---|---|
| CUDA OOM | Set `batch=8` |
| Low mAP | Increase `epochs=150`, use `yolov8s-seg.pt` |
| Too many false positives | Raise `conf=0.45` |
| Mask edges rough | Enable `retina_masks=True` |
| Dataset version wrong | Change `project.version(2)` |